### Comparación de Experimentos -- tema meses (Val-B2)

Compara el baseline Val-B2 (z803) con las hipótesis, a partir de los CSV en formato común de `out/` (`{experimento}_test`, `_cortes`, `_mejorN` y `_curvas`). Para cada experimento: mediana de las 5 semillas (con mínimo y máximo) y el semillerío aparte. Todos testean en 201910 con la ganancia contra BAJA+2.

#### Librerías y Path Variables

In [1]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle                                                                   # CAMBIO

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import roc_auc_score                                       # CAMBIO

import lightgbm as lgb                                                          # CAMBIO

import optuna                                                                   # CAMBIO
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour   # CAMBIO

In [4]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_meses.parquet'
ruta_meses = dataset_path + dataset_file

#### Carga de los Resultados

In [5]:
# Experimentos a comparar: el nombre es el de 'experimento' en cada notebook (prefijo de los CSV en out/)
experimentos = ['Baseline_ValB2', 'H01_ValB2']             # se agregan H2 y H3 cuando estén

def leer(tipo):
    # dtype str en 'modelo': las semillas y 'semillerio' quedan como texto, igual que al guardarlos
    return pd.concat([pd.read_csv(out_path + f'{e}_{tipo}.csv', dtype={'modelo': str}) for e in experimentos],
                     ignore_index=True)

comp_test, comp_cortes, comp_mejorN, comp_curvas = leer('test'), leer('cortes'), leer('mejorN'), leer('curvas')

# Chequeo: cada experimento tiene que tener 6 modelos (5 semillas + semillerío)
print(comp_test.groupby('experimento', sort=False)['modelo'].count())

experimento
Baseline_ValB2    6
H01_ValB2         6
Name: modelo, dtype: int64


##### Test con Umbral 0.025

In [11]:
from IPython.display import display

# 5 semillas: mediana, mínimo y máximo por experimento; el semillerío aparte (un solo modelo)
st = comp_test[comp_test['modelo'] != 'semillerio']
tabla_test = st.groupby('experimento', sort=False).agg(
    auc_mediana=('auc_test', 'median'), auc_min=('auc_test', 'min'), auc_max=('auc_test', 'max'),
    gan_mediana=('ganancia', 'median'), gan_min=('ganancia', 'min'), gan_max=('ganancia', 'max'),
)
sem = comp_test[comp_test['modelo'] == 'semillerio'].set_index('experimento')
tabla_test['auc_semillerio'] = sem['auc_test']
tabla_test['gan_semillerio'] = sem['ganancia']

cols_auc = ['auc_mediana', 'auc_min', 'auc_max', 'auc_semillerio']
cols_gan = ['gan_mediana', 'gan_min', 'gan_max', 'gan_semillerio']
tabla_test[cols_gan] = tabla_test[cols_gan] / 1e6                     # ganancia en millones
formato = {**{c: '{:.4f}' for c in cols_auc}, **{c: '{:.2f}' for c in cols_gan}}     # CAMBIO: un solo diccionario (un segundo .format pisa al primero)
display(tabla_test.style.format(formato)                                             # CAMBIO
        .set_caption('Test 201910 - AUC contra BAJA+2 y ganancia con umbral 0.025 (M): 5 semillas y semillerío'))

,auc_mediana,auc_min,auc_max,gan_mediana,gan_min,gan_max,auc_semillerio,gan_semillerio
experimento,,,,,,,,
Baseline_ValB2,0.9239,0.9229,0.9241,257.26,250.39,258.47,0.9242,260.45
H01_ValB2,0.9187,0.9175,0.9195,191.98,182.68,199.51,0.9198,193.66


##### Ganancia por Corte

In [7]:
# Mediana de las 5 semillas en cada corte fijo, un experimento por columna (millones)
sc = comp_cortes[comp_cortes['modelo'] != 'semillerio']
tabla_cortes = (sc.pivot_table(index='corte', columns='experimento', values='ganancia', aggfunc='median')
                  .reindex(columns=experimentos) / 1e6)
tabla_cortes.insert(0, '% clientes', sc.groupby('corte')['pct_clientes'].first())    # el % es el mismo en todos (octubre)
display(tabla_cortes.style.format('{:.2f}')
        .set_caption('Ganancia en Octubre por corte (M) - mediana de las 5 semillas'))

experimento,% clientes,Baseline_ValB2,H01_ValB2
corte,,,
2000,1.47,253.00,247.50
2500,1.84,272.25,263.45
3000,2.21,279.40,271.70
3500,2.58,288.75,277.75
4000,2.95,290.40,276.10
4500,3.32,290.95,277.75
5000,3.69,287.10,275.00
5500,4.06,283.25,271.15
6000,4.42,280.50,268.40


##### Boxplot por Corte

In [8]:
# Distribución de las 5 semillas en cada corte, un color por experimento: muestra si los rangos se superponen
fig = px.box(sc.assign(ganancia_M=sc['ganancia'] / 1e6), x='corte', y='ganancia_M', color='experimento',
             category_orders={'experimento': experimentos})
fig.update_layout(
    title='Ganancia por corte - 5 semillas por experimento',
    xaxis_title='Cantidad de clientes estimulados',
    yaxis_title='Ganancia en Octubre (M)',
    template='plotly_white'
)
fig.show()

##### Mejor Corte (usa las etiquetas de test: es un techo)

In [12]:
# Mediana de las 5 semillas del mejor corte y la ganancia máxima; el semillerío aparte
sm = comp_mejorN[comp_mejorN['modelo'] != 'semillerio']
tabla_mejorN = sm.groupby('experimento', sort=False)[['mejor_corte', 'pct_clientes', 'ganancia_maxima']].median()
semN = comp_mejorN[comp_mejorN['modelo'] == 'semillerio'].set_index('experimento')
tabla_mejorN['mejor_corte_semillerio'] = semN['mejor_corte']
tabla_mejorN['ganancia_maxima_semillerio'] = semN['ganancia_maxima']

cols_gan = ['ganancia_maxima', 'ganancia_maxima_semillerio']
tabla_mejorN[cols_gan] = tabla_mejorN[cols_gan] / 1e6                 # ganancia en millones
formato = {'mejor_corte': '{:,.0f}', 'mejor_corte_semillerio': '{:,.0f}', 'pct_clientes': '{:.2f}',
           **{c: '{:.2f}' for c in cols_gan}}                         # CAMBIO: un solo diccionario (un segundo .format pisa al primero)
display(tabla_mejorN.style.format(formato)                            # CAMBIO
        .set_caption('Mejor corte y ganancia máxima (M) - mediana de las 5 semillas y semillerío'))

,mejor_corte,pct_clientes,ganancia_maxima,mejor_corte_semillerio,ganancia_maxima_semillerio
experimento,,,,,
Baseline_ValB2,"4,170",3.07,293.15,"3,846",295.74
H01_ValB2,"4,282",3.16,279.40,"4,275",282.84


##### Curvas de Ganancia Acumulada

In [10]:
# Mediana de la ganancia acumulada de las 5 semillas, una línea por experimento
scu = comp_curvas[comp_curvas['modelo'] != 'semillerio']
curva_med = scu.groupby(['experimento', 'clientes'], sort=False)['ganancia_acumulada'].median().reset_index()
fig = px.line(curva_med, x='clientes', y='ganancia_acumulada', color='experimento',
              category_orders={'experimento': experimentos})
fig.update_layout(
    title='Ganancia acumulada en Octubre - mediana de 5 semillas',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)
fig.show()